In [1]:
import pandas as pd
df = pd.read_csv("../data/applications.csv")
df.head()

,loan_id,applied_month,bank_type,bank_id,grade,stars,card_ping_passed,first_draw,zero_pay_100d,paid_share_100d
0,100000,2025-11,traditional,T01,B,5,1,430.0,0,0.403
1,100001,2025-11,traditional,T07,B,5,1,440.0,0,0.950
2,100002,2025-11,traditional,T06,C,5,1,440.0,0,0.808
3,100003,2025-11,traditional,T12,B,5,1,390.0,1,0.000
4,100004,2025-11,traditional,T04,D,5,1,200.0,0,0.620


In [2]:
print(len(df), "rows")
print(df["loan_id"].nunique(), "unique loan_ids")
print(df.isna().sum().sum(), "missing values")

19900 rows
19900 unique loan_ids
0 missing values


In [3]:
by_month = df.groupby("applied_month")["zero_pay_100d"].agg(["count", "mean"]).round(3)
by_month

,count,mean
applied_month,,
2025-11,3400,0.282
2025-12,3900,0.282
2026-01,4600,0.322
2026-02,4900,0.351
2026-03,3100,0.377


In [4]:
share = pd.crosstab(df["applied_month"], df["bank_type"], normalize="index").round(3)
share

bank_type,neobank,other_flagged,traditional
applied_month,,,
2025-11,0.084,0.136,0.781
2025-12,0.099,0.167,0.734
2026-01,0.151,0.185,0.664
2026-02,0.189,0.211,0.600
2026-03,0.205,0.220,0.575


In [5]:
within = df.pivot_table(index="applied_month", columns="bank_type",
                        values="zero_pay_100d", aggfunc="mean").round(3)
within

bank_type,neobank,other_flagged,traditional
applied_month,,,
2025-11,0.479,0.430,0.236
2025-12,0.508,0.412,0.222
2026-01,0.563,0.443,0.233
2026-02,0.581,0.426,0.253
2026-03,0.640,0.494,0.238


In [6]:
nov_rates = within.loc["2025-11"]
mix_only = (share * nov_rates).sum(axis=1).round(3)
actual = by_month["mean"]
decomp = pd.DataFrame({"actual": actual,
                       "if_only_mix_changed": mix_only,
                       "composition_effect": (mix_only - actual.iloc[0]).round(3),
                       "within_group_effect": (actual - mix_only).round(3)})
decomp

,actual,if_only_mix_changed,composition_effect,within_group_effect
applied_month,,,,
2025-11,0.282,0.283,0.001,-0.001
2025-12,0.282,0.292,0.010,-0.010
2026-01,0.322,0.309,0.027,0.013
2026-02,0.351,0.323,0.041,0.028
2026-03,0.377,0.328,0.046,0.049


Zero pay moved up 9.5 points from November to March. Holding each bank type's November rate fixed and applying March's mix explains 4.6 points the portfolio shifted toward neobank applicants, who were already the riskiest group. The remaining 4.9 points is real deterioration, and it didn't begin until January; December's groups were flat or slightly better. The within-group table shows where it lives: traditional-bank rates barely moved all window, while neobank rates climbed from 48% to 64%. So zero pay went up is two problems, roughly equal in size, with different fixes one is who we're funding, the other is that one group got worse.

In [9]:
cells = df.groupby(["bank_id", "grade", "stars"]).size()
print(len(cells), "cells;", (cells < 30).sum(), "with fewer than 30 loans")

400 cells; 170 with fewer than 30 loans


400 cells; 170 with fewer than 30 loans

Build window: November through February. Holdout: March. March is the most recent month, the smallest, and the one where within-group deterioration is largest — it's the honest test of whether a score built on earlier months survives the month that behaved differently.